## Import Python Libraries and load environment variables

In [ ]:
import os
import json
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

In [ ]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

## Create a helper function to beautify llm responses in our Jupyter Notebook

In [ ]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [ ]:
# bfmsgs(result)

## Create a dummy function that returns 1500 as KRW > USD FX

In [ ]:
def get_fx(currency: str) -> float:
    """Return conversion rate from KRW to the given currency (e.g., USD)."""
    # Read API key from environment variables loaded by load_dotenv()
    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing EXCHANGE_RATE_API_KEY in .env")

    # Build ExchangeRate-API endpoint with KRW as base currency
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/KRW"

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; hai5016-project/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(f"Failed to fetch exchange rates: {error}") from error

    # Optional API-level error check
    if data.get("result") != "success":
        error_type = data.get("error-type", "unknown_error")
        raise RuntimeError(f"ExchangeRate API error: {error_type}")

    rates = data.get("conversion_rates", {})
    code = currency.upper()

    if code not in rates:
        raise ValueError(f"Currency code not found: {code}")

    return float(rates[code])

In [ ]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [ ]:
from datetime import datetime
import re

@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch menu-like text blocks from a URL and keep only useful content."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as error:
        return f"Fetch failed: {error}"

    # Decode HTML safely
    html = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(html, "html.parser")

    # Remove non-content tags that add a lot of noise
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
        tag.decompose()

    # Keep likely menu containers first (table/list/sections with diet/menu keywords)
    candidate_blocks = []
    keyword_pattern = re.compile(r"(식단|메뉴|중식|석식|조식|lunch|dinner|breakfast|menu)", re.IGNORECASE)

    selectors = [
        "table",
        "ul",
        "ol",
        "section",
        "article",
        "div",
    ]

    for selector in selectors:
        for node in soup.select(selector):
            text = node.get_text(" ", strip=True)
            if len(text) < 30:
                continue
            if keyword_pattern.search(text):
                candidate_blocks.append(text)

    # Fallback to body text if no candidate block is found
    if not candidate_blocks and soup.body:
        candidate_blocks = [soup.body.get_text(" ", strip=True)]

    # De-duplicate and keep output small enough for the model
    unique_blocks = []
    seen = set()
    for block in candidate_blocks:
        normalized = " ".join(block.split())
        if normalized not in seen:
            seen.add(normalized)
            unique_blocks.append(normalized)

    # Include today's date to improve downstream filtering
    today = datetime.now()
    today_hint = (
        f"TODAY_HINT: {today.strftime('%Y-%m-%d')} / "
        f"{today.strftime('%m/%d')} / "
        f"{today.strftime('%-m/%-d')}"
    )

    trimmed_text = "\n\n".join(unique_blocks[:20])
    return f"SOURCE_URL: {url}\n{today_hint}\n\n{trimmed_text[:25000]}"

In [ ]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant for Busan university cafeterias.
Always answer in {language}.

>>> CRITICAL TASK <<<
Return a tight pick-from-this-list menu for TODAY only.
Do not write a broad campus summary.

>>> REQUIRED WORKFLOW <<<
1) Use fetch_text_from_url for each provided URL.
2) Extract only entries that are explicitly for today using date clues (TODAY_HINT, weekday names, or date strings).
3) If a page is weekly, keep only today's row/items.
4) If today cannot be confirmed for a campus, output exactly: No confirmed today menu found.

>>> OUTPUT FORMAT (STRICT) <<<
Title: Today Lunch Picks (Busan)
Then a numbered list with at most 8 items.
Each item must be one line and follow this format exactly:
{{Campus}} | {{Meal name}} | {{Price KRW or N/A}} | {{Meal time or N/A}} | {{Source URL}}

>>> CONTENT RULES <<<
- Include only lunch-relevant options when available.
- Translate Korean meal names to English in parentheses if needed.
- Never include generic statements like 'typically offered' or 'may vary'.
- Never include analysis paragraphs before or after the list."""

In [ ]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url, get_fx],
    system_prompt=SYSTEM_PROMPT
)

- https://www.hanyang.ac.kr/re12
- http://hfspn.co/

In [ ]:
today_text = "today"
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Fetch menu data from these URLs and return ONLY a tight pick-from-this-list for today lunch in Busan. "
                    "Do not summarize. If today is not identifiable for a campus, write: No confirmed today menu found. "
                    "URLs: "
                    "https://www.pusan.ac.kr/kor/CMS/MenuMgr/menuListOnWeekly.do?mCode=MN203, "
                    "https://www.pknu.ac.kr/main/399?action=view&no=724732, "
                    "https://www.donga.ac.kr/kor/CMS/DietMenuMgr/list.do?mCode=MN199, "
                    "https://dorm.deu.ac.kr/60/6050.do, "
                    "https://www.bufs.ac.kr/bbs/board.php?bo_table=weekly_meal&wr_id=1, "
                    "https://www.kmou.ac.kr/coop/dv/dietView/selectDietDateView.do?mi=1189"
                ),
            }
        ]
    }
)
bfmsgs(result)

In [ ]:
# Diagnostics: did the agent call tools, and how many URLs were fetched?
messages = result.get('messages', [])
print('Total messages:', len(messages))

for index, message in enumerate(messages):
    message_type = type(message).__name__
    print(f'[{index}] {message_type}')
    if message_type == 'AIMessage':
        tool_calls = getattr(message, 'tool_calls', None)
        if tool_calls:
            print('  tool_calls:', len(tool_calls))
            for call in tool_calls:
                print('   -', call.get('name'), call.get('args'))
    if message_type == 'ToolMessage':
        content_preview = str(getattr(message, 'content', ''))[:140]
        print('  tool_output_preview:', content_preview.replace('\n', ' '))